# E3 — Truncated Newton Kernel Ridge Regression

Implementación separada del KRR de solución exacta. El artículo [aplicado de Abdulla et al. (2022)](https://doi.org/10.3390/app12094252) remite al método TR-KRR de [Maalouf y Homouz (2014)](https://doi.org/10.1016/j.knosys.2014.08.012).

## Procedimiento

El estimador resuelve `(K + alpha I) a = y` para el kernel RBF `exp(-gamma ||x-z||²)` con gradiente conjugado lineal desde `a=0`. Usa máximo 200 iteraciones, tolerancia de norma residual 0,5 y terminación después de 25 iteraciones sin mejora, como publica Maalouf y Homouz. El artículo no precisa la métrica de no mejora ni el vector inicial: usamos residuo L2 y `a=0`. No transforma `y`.

El protocolo de datos es la adaptación E3: train/test temporal canónico, folds agrupados por estudiante, pipeline ajustado dentro de cada fit, grilla alpha/gamma de 35 candidatos y selección por RMSE anidada 5×5. Test no elige parámetros. Esto implementa el solver del paper, sin afirmar una réplica de su experimento.

In [ ]:
from pathlib import Path
import sys
BOOK = Path.cwd() if (Path.cwd() / 'evaluar_tn_krr_e3.py').exists() else Path.cwd() / 'Precalculo'
sys.path.insert(0, str(BOOK.resolve()))
import pandas as pd
from evaluar_tn_krr_e3 import run, ROOT
from tn_krr_e3 import TruncatedNewtonKernelRidgeRegressor
report = run()
display(pd.DataFrame(report["cv"]).round(6))
print("Parámetros finales:", report["best_params"])

## Test temporal y diagnóstico del optimizador

In [ ]:
display(pd.DataFrame([report["test"]]).round(6))
display(pd.DataFrame(report["fold_audit"]))
grid = pd.read_csv(ROOT / "grid_final.csv")
display(grid.groupby(["alpha", "gamma"])[
    ["RMSE", "cg_iterations", "cg_residual_norm", "cg_converged"]
].mean().sort_values("RMSE").head(10))

El informe separa la media de los folds externos, OOF agrupado y test. Un solver que se detenga por máximo de iteraciones o por no mejora devuelve la aproximación TR-CG y queda identificado en la auditoría.